# Notebook 04 — Statistical Analysis
**Project:** Ola Bengaluru Rides — Cancellation Intelligence & Revenue Recovery  
**Dataset:** Bengaluru Ola Rides, January 2024  
**Author:** Joshit  
**Institute:** Newton School of Technology — DVA Capstone 2

EDA showed us the patterns. This notebook proves whether those patterns are statistically significant or just random noise. Covers correlation, hypothesis testing, regression and cancellation risk scoring.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, f_oneway, ttest_ind, pearsonr
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('ready.')

In [ ]:
# update path if running locally: '../data/processed/ola_bengaluru_cleaned.csv'
CLEAN_PATH = '/content/ola_bengaluru_cleaned.csv'

df = pd.read_csv(CLEAN_PATH)
df['date'] = pd.to_datetime(df['date'])
df_success = df[df['is_successful'] == 1].copy()

print(f'loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'successful rides for numeric analysis: {len(df_success):,}')

### Correlation Analysis
Starting with a correlation heatmap to understand which numeric variables are related to each other. This gives direction for the hypothesis tests below.

In [ ]:
num_cols = ['booking_value', 'ride_distance', 'avg_vtat', 'avg_ctat', 'driver_rating', 'customer_rating']
corr_matrix = df_success[num_cols].corr()

plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, vmin=-1, vmax=1, linewidths=0.5,
            cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Correlation Matrix — Numeric Variables (Successful Rides)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/04_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nstrong correlations (|r| > 0.3):')
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.3:
            print(f'  {corr_matrix.columns[i]} vs {corr_matrix.columns[j]}: r = {r:.3f}')

### Hypothesis Test 1 — Chi-Square Test
**Question:** Is there a statistically significant relationship between vehicle type and cancellation?

- H0 (Null): Vehicle type and cancellation status are independent — no relationship
- H1 (Alternative): Vehicle type and cancellation status are dependent — there is a relationship
- Significance level: 0.05

In [ ]:
contingency = pd.crosstab(df['vehicle_type'], df['is_cancelled'])
print('contingency table (vehicle type vs cancelled):')
print(contingency)

chi2, p_value, dof, expected = chi2_contingency(contingency)

print(f'\nChi-Square Statistic : {chi2:.4f}')
print(f'P-Value              : {p_value:.6f}')
print(f'Degrees of Freedom   : {dof}')
print()
if p_value < 0.05:
    print('Result: REJECT H0 — vehicle type and cancellation are statistically dependent')
    print('Interpretation: The type of vehicle significantly affects whether a ride gets cancelled')
else:
    print('Result: FAIL TO REJECT H0 — no significant relationship found')

In [ ]:
cancel_by_vehicle = df.groupby('vehicle_type')['is_cancelled'].mean() * 100

plt.figure(figsize=(10, 5))
colors = ['#E74C3C' if v > cancel_by_vehicle.mean() else '#3498DB' for v in cancel_by_vehicle.values]
bars = plt.bar(cancel_by_vehicle.index, cancel_by_vehicle.values, color=colors, edgecolor='white')
plt.axhline(cancel_by_vehicle.mean(), color='black', linestyle='--',
            linewidth=1.2, label=f'avg: {cancel_by_vehicle.mean():.1f}%')
plt.title('Cancellation Rate by Vehicle Type\n(Chi-Square p-value: ' + f'{p_value:.4f})', fontsize=12, fontweight='bold')
plt.ylabel('Cancellation Rate (%)')
plt.legend()
for bar, val in zip(bars, cancel_by_vehicle.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{val:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('/content/04_chi_square_vehicle.png', dpi=150, bbox_inches='tight')
plt.show()

### Hypothesis Test 2 — Independent Samples T-Test
**Question:** Is the average VTAT (vehicle arrival time) significantly higher in high-cancellation areas compared to low-cancellation areas?

- H0: No significant difference in VTAT between high and low cancellation areas
- H1: High cancellation areas have significantly higher VTAT
- Significance level: 0.05

In [ ]:
area_cancel_rate = df.groupby('pickup_location')['is_cancelled'].mean()
median_rate = area_cancel_rate.median()

high_cancel_areas = area_cancel_rate[area_cancel_rate >= median_rate].index
low_cancel_areas = area_cancel_rate[area_cancel_rate < median_rate].index

high_vtat = df_success[df_success['pickup_location'].isin(high_cancel_areas)]['avg_vtat'].dropna()
low_vtat = df_success[df_success['pickup_location'].isin(low_cancel_areas)]['avg_vtat'].dropna()

t_stat, p_value_ttest = ttest_ind(high_vtat, low_vtat)

print(f'High cancellation areas — Avg VTAT: {high_vtat.mean():.2f} min (n={len(high_vtat):,})')
print(f'Low cancellation areas  — Avg VTAT: {low_vtat.mean():.2f} min (n={len(low_vtat):,})')
print(f'\nT-Statistic : {t_stat:.4f}')
print(f'P-Value     : {p_value_ttest:.6f}')
print()
if p_value_ttest < 0.05:
    print('Result: REJECT H0 — VTAT is significantly different between high and low cancellation areas')
    print('Interpretation: Longer vehicle arrival times are statistically linked to higher cancellation rates')
else:
    print('Result: FAIL TO REJECT H0 — no significant VTAT difference found')

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(high_vtat, bins=30, alpha=0.6, color='#E74C3C', label=f'High cancel areas (mean={high_vtat.mean():.1f} min)')
plt.hist(low_vtat, bins=30, alpha=0.6, color='#3498DB', label=f'Low cancel areas (mean={low_vtat.mean():.1f} min)')
plt.title('VTAT Distribution — High vs Low Cancellation Areas\n(T-Test p-value: ' + f'{p_value_ttest:.4f})', fontsize=12, fontweight='bold')
plt.xlabel('Avg VTAT (minutes)')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.savefig('/content/04_ttest_vtat.png', dpi=150, bbox_inches='tight')
plt.show()

### Hypothesis Test 3 — One-Way ANOVA
**Question:** Do cancellation rates differ significantly across different times of day?

- H0: Cancellation rates are equal across all time of day groups
- H1: At least one time of day group has a significantly different cancellation rate
- Significance level: 0.05

In [ ]:
time_groups = [group['is_cancelled'].values
               for name, group in df.groupby('time_of_day')]

f_stat, p_value_anova = f_oneway(*time_groups)

print('cancellation rate by time of day:')
tod_cancel = df.groupby('time_of_day')['is_cancelled'].mean() * 100
for tod, rate in tod_cancel.sort_values(ascending=False).items():
    print(f'  {tod:<20} {rate:.2f}%')

print(f'\nF-Statistic : {f_stat:.4f}')
print(f'P-Value     : {p_value_anova:.6f}')
print()
if p_value_anova < 0.05:
    print('Result: REJECT H0 — cancellation rates differ significantly across time of day')
    print('Interpretation: Time of day has a statistically significant effect on cancellation probability')
else:
    print('Result: FAIL TO REJECT H0 — no significant difference across time groups')

In [ ]:
order = ['Early Morning', 'Morning', 'Afternoon', 'Evening', 'Night', 'Late Night']
tod_cancel_ordered = tod_cancel.reindex(order)

plt.figure(figsize=(10, 5))
bars = plt.bar(tod_cancel_ordered.index, tod_cancel_ordered.values, color='#9B59B6', edgecolor='white')
plt.axhline(tod_cancel_ordered.mean(), color='black', linestyle='--',
            linewidth=1.2, label=f'avg: {tod_cancel_ordered.mean():.1f}%')
plt.title('Cancellation Rate by Time of Day\n(ANOVA p-value: ' + f'{p_value_anova:.4f})', fontsize=12, fontweight='bold')
plt.ylabel('Cancellation Rate (%)')
plt.legend()
for bar, val in zip(bars, tod_cancel_ordered.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{val:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('/content/04_anova_timeofday.png', dpi=150, bbox_inches='tight')
plt.show()

### Linear Regression — Ride Distance vs Booking Value
**Question:** Does ride distance predict booking value? How strong is the relationship?

Using successful rides only since cancelled rides have no booking value.

In [ ]:
reg_df = df_success[['ride_distance', 'booking_value']].dropna()

X = reg_df[['ride_distance']].values
y = reg_df['booking_value'].values

model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
pearson_r, pearson_p = pearsonr(reg_df['ride_distance'], reg_df['booking_value'])

print(f'Slope (fare per km)  : ₹{model.coef_[0]:.2f}')
print(f'Intercept            : ₹{model.intercept_:.2f}')
print(f'R² Score             : {r2:.4f}')
print(f'RMSE                 : ₹{rmse:.2f}')
print(f'Pearson r            : {pearson_r:.4f}')
print(f'Pearson p-value      : {pearson_p:.6f}')
print()
print(f'Interpretation: For every 1 km increase in ride distance, booking value increases by ₹{model.coef_[0]:.2f}')
print(f'R² of {r2:.2f} means ride distance explains {r2*100:.1f}% of variation in booking value')

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(reg_df['ride_distance'], reg_df['booking_value'],
            alpha=0.3, color='#3498DB', s=10, label='Actual rides')
plt.plot(reg_df['ride_distance'], y_pred, color='#E74C3C',
         linewidth=2, label=f'Regression line (R²={r2:.3f})')
plt.title('Ride Distance vs Booking Value — Linear Regression', fontsize=13, fontweight='bold')
plt.xlabel('Ride Distance (km)')
plt.ylabel('Booking Value (₹)')
plt.legend()
plt.tight_layout()
plt.savefig('/content/04_regression_plot.png', dpi=150, bbox_inches='tight')
plt.show()

### Distribution Analysis
Checking if booking value and ride distance follow a normal distribution — important for validating the statistical tests above.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0,0].hist(df_success['booking_value'].dropna(), bins=40, color='#2ECC71', edgecolor='white')
axes[0,0].set_title('Booking Value Distribution', fontweight='bold')
axes[0,0].set_xlabel('Booking Value (₹)')
axes[0,0].axvline(df_success['booking_value'].mean(), color='red', linestyle='--',
                   label=f'mean: ₹{df_success["booking_value"].mean():.0f}')
axes[0,0].legend()

axes[0,1].hist(df_success['ride_distance'].dropna(), bins=40, color='#3498DB', edgecolor='white')
axes[0,1].set_title('Ride Distance Distribution', fontweight='bold')
axes[0,1].set_xlabel('Distance (km)')
axes[0,1].axvline(df_success['ride_distance'].mean(), color='red', linestyle='--',
                   label=f'mean: {df_success["ride_distance"].mean():.1f} km')
axes[0,1].legend()

stats.probplot(df_success['booking_value'].dropna(), dist='norm', plot=axes[1,0])
axes[1,0].set_title('Booking Value — Q-Q Plot', fontweight='bold')

stats.probplot(df_success['ride_distance'].dropna(), dist='norm', plot=axes[1,1])
axes[1,1].set_title('Ride Distance — Q-Q Plot', fontweight='bold')

plt.suptitle('Distribution Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/04_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### Cancellation Risk Scoring
Combining vehicle type + time of day to create a risk score for each combination. This tells operations exactly which slots need intervention.

In [ ]:
risk = df.groupby(['vehicle_type', 'time_of_day']).agg(
    total=('booking_id', 'count'),
    cancelled=('is_cancelled', 'sum')
).reset_index()

risk['cancel_rate'] = (risk['cancelled'] / risk['total'] * 100).round(2)

overall_mean = risk['cancel_rate'].mean()
overall_std = risk['cancel_rate'].std()
risk['risk_score'] = ((risk['cancel_rate'] - overall_mean) / overall_std).round(2)

risk['risk_label'] = risk['risk_score'].apply(
    lambda x: 'High Risk' if x > 1 else ('Low Risk' if x < -1 else 'Medium Risk'))

print('top 10 highest risk combinations:')
print(risk.sort_values('risk_score', ascending=False).head(10)[[
    'vehicle_type', 'time_of_day', 'total', 'cancelled', 'cancel_rate', 'risk_score', 'risk_label'
]].to_string(index=False))

In [ ]:
order = ['Early Morning', 'Morning', 'Afternoon', 'Evening', 'Night', 'Late Night']
risk_pivot = risk.pivot(index='vehicle_type', columns='time_of_day', values='cancel_rate')
risk_pivot = risk_pivot.reindex(columns=[c for c in order if c in risk_pivot.columns])

plt.figure(figsize=(13, 6))
sns.heatmap(risk_pivot, annot=True, fmt='.1f', cmap='RdYlGn_r',
            linewidths=0.5, cbar_kws={'label': 'Cancellation Rate (%)'})
plt.title('Cancellation Risk — Vehicle Type vs Time of Day', fontsize=13, fontweight='bold')
plt.xlabel('Time of Day')
plt.ylabel('Vehicle Type')
plt.tight_layout()
plt.savefig('/content/04_risk_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### Revenue Recovery Estimate
Using actual data to quantify how much revenue can be recovered if cancellation rate drops by 15%.

In [ ]:
total_rides = len(df)
total_cancelled = df['is_cancelled'].sum() + (df['booking_status'] == 'Incomplete').sum()
current_cancel_rate = total_cancelled / total_rides * 100
avg_fare = df_success['booking_value'].mean()

target_reduction = 0.15
rides_recovered = int(total_cancelled * target_reduction)
revenue_recovered = rides_recovered * avg_fare

print('=== Revenue Recovery Estimate ===')
print(f'Total rides            : {total_rides:,}')
print(f'Non-successful rides   : {total_cancelled:,}')
print(f'Current cancel rate    : {current_cancel_rate:.1f}%')
print(f'Avg fare (successful)  : ₹{avg_fare:.2f}')
print()
print(f'Target reduction       : 15%')
print(f'Rides recovered        : {rides_recovered:,}')
print(f'Revenue recovered      : ₹{revenue_recovered:,.0f}')
print(f'Revenue recovered (L)  : ₹{revenue_recovered/100000:.1f} Lakh')
print()

# by vehicle type
print('revenue recovery by vehicle type:')
veh_cancelled = df[df['is_cancelled'] == 1].groupby('vehicle_type').size()
for veh, count in veh_cancelled.items():
    recovered = int(count * target_reduction) * avg_fare
    print(f'  {veh:<15} {int(count * target_reduction):>4} rides recovered  ₹{recovered:>10,.0f}')

In [ ]:
veh_recovery = {}
for veh, count in veh_cancelled.items():
    veh_recovery[veh] = int(count * target_reduction) * avg_fare

plt.figure(figsize=(10, 5))
bars = plt.bar(veh_recovery.keys(), [v/100000 for v in veh_recovery.values()],
               color='#2ECC71', edgecolor='white')
plt.title('Estimated Revenue Recovery by Vehicle Type\n(at 15% Cancellation Reduction)', fontsize=12, fontweight='bold')
plt.ylabel('Revenue Recovery (₹ Lakh)')
plt.xlabel('Vehicle Type')
for bar, val in zip(bars, veh_recovery.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'₹{val/100000:.1f}L', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('/content/04_revenue_recovery.png', dpi=150, bbox_inches='tight')
plt.show()

### Statistical findings summary

| Test | Question | Result | Business Meaning |
|---|---|---|---|
| Chi-Square | Does vehicle type affect cancellation? | Check p-value above | Vehicle choice is/isn't statistically linked to cancellations |
| T-Test | Is VTAT higher in high-cancel areas? | Check p-value above | Longer wait times do/don't cause more cancellations |
| ANOVA | Does time of day affect cancellation rate? | Check p-value above | Time of day does/doesn't significantly change cancellation risk |
| Regression | Does distance predict fare? | Check R² above | Distance explains X% of fare variation |

These results directly support the business recommendations in the final report.

In [ ]:
# saving risk scores
risk.to_csv('/content/04_risk_scores.csv', index=False)

# saving statistical summary
stat_summary = pd.DataFrame({
    'Test': ['Chi-Square', 'T-Test (VTAT)', 'ANOVA (Time of Day)', 'Linear Regression'],
    'Statistic': [f'{chi2:.4f}', f'{t_stat:.4f}', f'{f_stat:.4f}', f'R²={r2:.4f}'],
    'P-Value': [f'{p_value:.6f}', f'{p_value_ttest:.6f}', f'{p_value_anova:.6f}', f'{pearson_p:.6f}'],
    'Significant': [
        'Yes' if p_value < 0.05 else 'No',
        'Yes' if p_value_ttest < 0.05 else 'No',
        'Yes' if p_value_anova < 0.05 else 'No',
        'Yes' if pearson_p < 0.05 else 'No'
    ]
})
stat_summary.to_csv('/content/04_statistical_summary.csv', index=False)

print('saved.')
print(stat_summary.to_string(index=False))
print('\nproceed to 05_final_load_prep.ipynb')